# FATS WORLD Suppression Gap Validation

Checks consistency between the Eurostat FATS `WORLD` aggregate and the sum of
individual real-country rows for the `B-S_X_O` (Total business sector) industry code.

**Rationale:** Eurostat suppresses country-level data when values are confidential
(small cell sizes, dominant firms). The `WORLD` total is reported even when individual
country rows are suppressed (NaN). The gap `WORLD - sum(countries)` quantifies the
share of activity that is "hidden" by suppression and cannot be recovered at the
country level.

In [1]:
import os
import pandas as pd
import numpy as np

# Path resolution: notebook lives in notebooks/, so ROOT = one level up
ROOT   = os.path.normpath(os.path.join(os.getcwd(), ".."))
SILVER = os.path.join(ROOT, "data", "Silver")

df = pd.read_parquet(os.path.join(SILVER, "fats_clean.parquet"))
print(f"Loaded: {df.shape}")
print(f"Host countries in file: {df['host_country'].nunique()}")
print(f"NACE codes present: {sorted(df['nace_r2_section'].unique())}")

Loaded: (1325218, 7)
Host countries in file: 255
NACE codes present: ['A', 'B', 'B-S_X_O', 'C', 'E', 'F', 'G', 'G-S_X_O', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S']


In [2]:
# All non-country host entities in the FATS data
GEO_AGGREGATES = {
    "WORLD",
    "EU25", "EU27_2007", "EU27_2020", "EU28", "EA13",
    "EXT_EU25", "EXT_EU27_2007", "EXT_EU27_2020", "EXT_EU28",
    "EXT_EU_NAL", "EXT_EA13",
    "OFFSHO",
    "AFR", "AME", "ASI", "EUR", "OCE_PLR",
    "AN",   # Netherlands Antilles (dissolved 2010)
}

# Filter to B-S_X_O (Total business sector) — only aggregate with complete WORLD coverage
df_bso = df[df["nace_r2_section"] == "B-S_X_O"].copy()

df_world     = df_bso[df_bso["host_country"] == "WORLD"]
df_countries = df_bso[~df_bso["host_country"].isin(GEO_AGGREGATES)]

print(f"B-S_X_O rows total    : {len(df_bso):,}")
print(f"  → WORLD rows        : {len(df_world):,}")
print(f"  → Real-country rows : {len(df_countries):,}")
print(f"  → Other aggregate rows: {len(df_bso) - len(df_world) - len(df_countries):,}")

B-S_X_O rows total    : 255,722
  → WORLD rows        : 936
  → Real-country rows : 241,820
  → Other aggregate rows: 12,966


In [3]:
# Sum real countries per [year, home_country, variable]
# min_count=1 → NaN if ALL inputs are NaN (avoids treating all-suppressed as 0)
sum_ctry = (
    df_countries
    .groupby(["year", "home_country", "variable"], as_index=False)["value"]
    .sum(min_count=1)
    .rename(columns={"value": "sum_countries"})
)

world_vals = (
    df_world[["year", "home_country", "variable", "value"]]
    .rename(columns={"value": "world_value"})
)

cmp = world_vals.merge(sum_ctry, on=["year", "home_country", "variable"], how="left")
cmp["gap"]             = cmp["world_value"] - cmp["sum_countries"]
cmp["suppression_pct"] = 100 * cmp["gap"] / cmp["world_value"]

print(f"Comparison rows: {len(cmp):,}")
cmp.head(10)

Comparison rows: 936


,year,home_country,variable,world_value,sum_countries,gap,suppression_pct
0,2010,AUT,EMP,561613.0,560039.0,1574.0,0.280264
1,2010,AUT,ENT,3756.0,3756.0,0.0,0.000000
2,2010,AUT,TUR,121188.0,120775.0,413.0,0.340793
3,2011,AUT,EMP,571812.0,570284.0,1528.0,0.267221
4,2011,AUT,ENT,3993.0,3993.0,0.0,0.000000
5,2011,AUT,TUR,138528.0,138032.0,496.0,0.358050
6,2012,AUT,EMP,570824.0,569100.0,1724.0,0.302020
7,2012,AUT,ENT,4114.0,4114.0,0.0,0.000000
8,2012,AUT,TUR,147855.0,147196.0,659.0,0.445707
9,2013,AUT,EMP,567521.0,564578.0,2943.0,0.518571


## Per-year suppression summary

Values are averaged across all `home_country × variable` combinations per year.

- `world_value` — mean WORLD total reported by Eurostat
- `sum_countries` — mean sum of individual real-country rows
- `gap` — mean absolute difference (suppressed activity)
- `suppression_pct` — mean share of WORLD value that is unobservable at country level

In [4]:
yearly = (
    cmp.groupby("year")[["world_value", "sum_countries", "gap", "suppression_pct"]]
    .mean()
    .round(2)
)
display(yearly)

,world_value,sum_countries,gap,suppression_pct
year,,,,
2008,248287.55,210286.18,4029.52,4.35
2009,371660.09,317592.05,25626.96,14.82
2010,357501.44,337898.59,15270.81,11.08
2011,372246.19,356615.97,11058.22,10.53
2012,393120.95,374419.05,13901.65,12.28
2013,404875.62,376551.78,23496.24,13.16
2014,387904.24,360618.87,27285.37,13.82
2015,400482.00,373141.56,27340.44,15.48
2016,420914.89,393906.96,27007.92,14.62


## Per-variable suppression summary

ENT (enterprises), EMP (employment), TUR (turnover) may have different suppression rates
because Eurostat applies confidentiality rules independently per indicator.

In [5]:
by_var = (
    cmp.groupby("variable")[["world_value", "sum_countries", "gap", "suppression_pct"]]
    .mean()
    .round(2)
)
display(by_var)

,world_value,sum_countries,gap,suppression_pct
variable,,,,
EMP,912121.69,843521.76,54763.91,16.43
ENT,6537.99,6312.09,90.39,5.93
TUR,260424.41,240532.91,12535.54,17.16


In [6]:
# Overall stats
print(f"Median suppression rate across all home×year×variable : {cmp['suppression_pct'].median():.1f}%")
print(f"Mean suppression rate                                 : {cmp['suppression_pct'].mean():.1f}%")
print(f"Max suppression rate                                  : {cmp['suppression_pct'].max():.1f}%")
print()
# Flag any rows where real-country sum exceeds WORLD (Eurostat inconsistency / rounding)
n_inconsistent = (cmp["sum_countries"] > cmp["world_value"]).sum()
print(f"Rows where sum_countries > world_value (inconsistency): {n_inconsistent}")
if n_inconsistent > 0:
    display(cmp[cmp["sum_countries"] > cmp["world_value"]][["year","home_country","variable",
                                                              "world_value","sum_countries","gap"]])

Median suppression rate across all home×year×variable : 1.6%
Mean suppression rate                                 : 13.1%
Max suppression rate                                  : 100.0%

Rows where sum_countries > world_value (inconsistency): 10


,year,home_country,variable,world_value,sum_countries,gap
192,2010,GRC,TUR,8523.0,8524.0,-1.0
312,2010,HRV,TUR,2409.0,2411.0,-2.0
672,2010,SVN,TUR,6851.0,6853.0,-2.0
675,2011,SVN,TUR,8023.0,8024.0,-1.0
681,2013,SVN,TUR,7296.0,7300.0,-4.0
687,2015,SVN,TUR,5770.0,5772.0,-2.0
691,2017,SVN,EMP,35905.0,35907.0,-2.0
697,2019,SVN,EMP,34008.0,34010.0,-2.0
699,2019,SVN,TUR,6952.0,6955.0,-3.0
824,2008,GRC,TUR,10368.0,10369.0,-1.0


In [7]:
# Worst-suppressed home countries (mean suppression % across years × variables)
by_home = (
    cmp.groupby("home_country")["suppression_pct"]
    .mean()
    .sort_values(ascending=False)
    .round(1)
    .rename("mean_suppression_pct")
    .reset_index()
)
print("Top 10 home countries by mean suppression rate:")
display(by_home.head(10))

Top 10 home countries by mean suppression rate:


,home_country,mean_suppression_pct
0,MLT,75.5
1,CYP,44.5
2,LUX,41.5
3,NOR,31.4
4,SWE,28.5
5,IRL,19.9
6,LVA,18.6
7,POL,17.0
8,BEL,13.2
9,SVK,12.3


In [8]:
# Save detailed comparison to CSV for reference
out_csv = os.path.join(SILVER, "fats_world_suppression.csv")
cmp.to_csv(out_csv, index=False)
print(f"Saved detailed table: {out_csv}")
print(f"Rows: {len(cmp):,}  |  Columns: {list(cmp.columns)}")

Saved detailed table: <repo>\data\Silver\fats_world_suppression.csv
Rows: 936  |  Columns: ['year', 'home_country', 'variable', 'world_value', 'sum_countries', 'gap', 'suppression_pct']
